In [3]:
import numpy as np

# ==========================================
# 1. CONVOLUTIONAL LAYER (Conv2D)
# ==========================================
class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size, custom_weights=None, custom_bias=None):
        self.kernel_size = kernel_size
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        if custom_weights is not None:
            self.weights = custom_weights.astype(float)
        else:
            self.weights = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.1
            
        if custom_bias is not None:
            self.biases = custom_bias.astype(float)
        else:
            self.biases = np.zeros((out_channels, 1), dtype=float)
            
        self.dweights = None
        self.dbiases = None

    def forward(self, X):
        self.X = X
        batch_size, in_channels, in_height, in_width = X.shape
        _, _, k_h, k_w = self.weights.shape
        
        out_height = in_height - k_h + 1
        out_width = in_width - k_w + 1
        
        output = np.zeros((batch_size, self.out_channels, out_height, out_width))

        for b in range(batch_size):
            for c_out in range(self.out_channels):
                for h in range(out_height):
                    for w in range(out_width):
                        patch = X[b, :, h:h+k_h, w:w+k_w]
                        output[b, c_out, h, w] = (np.sum(patch * self.weights[c_out]) + self.biases[c_out]).item()
                        
        return output

    def backward(self, d_out, learning_rate=0.01):
        batch_size, _, out_height, out_width = d_out.shape
        _, _, k_h, k_w = self.weights.shape
        
        dX = np.zeros_like(self.X, dtype=float)
        self.dweights = np.zeros_like(self.weights, dtype=float)
        self.dbiases = np.zeros_like(self.biases, dtype=float)

        for b in range(batch_size):
            for c_out in range(self.out_channels):
                for h in range(out_height):
                    for w in range(out_width):
                        patch = self.X[b, :, h:h+k_h, w:w+k_w]
                        self.dweights[c_out] += patch * d_out[b, c_out, h, w]
                        dX[b, :, h:h+k_h, w:w+k_w] += self.weights[c_out] * d_out[b, c_out, h, w]
                self.dbiases[c_out] += np.sum(d_out[b, c_out, :, :])

        self.weights -= learning_rate * self.dweights
        self.biases -= learning_rate * self.dbiases

        return dX


# ==========================================
# 2. RELU ACTIVATION LAYER
# ==========================================
class ReLU:
    def forward(self, X):
        self.X = X
        return np.maximum(0, X)

    def backward(self, d_out):
        return d_out * (self.X > 0)


# ==========================================
# 3. MAX POOLING LAYER (MaxPool2D)
# ==========================================
class MaxPool2D:
    def __init__(self, pool_size=2, stride=1):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, X):
        self.X = X
        batch_size, channels, in_height, in_width = X.shape
        p_size = self.pool_size
        
        out_height = (in_height - p_size) // self.stride + 1
        out_width = (in_width - p_size) // self.stride + 1
        
        output = np.zeros((batch_size, channels, out_height, out_width))

        for b in range(batch_size):
            for c in range(channels):
                for h in range(out_height):
                    for w in range(out_width):
                        h_start = h * self.stride
                        h_end = h_start + p_size
                        w_start = w * self.stride
                        w_end = w_start + p_size
                        
                        patch = X[b, c, h_start:h_end, w_start:w_end]
                        output[b, c, h, w] = np.max(patch)
                        
        return output

    def backward(self, d_out):
        batch_size, channels, in_height, in_width = self.X.shape
        dX = np.zeros_like(self.X, dtype=float)
        p_size = self.pool_size

        for b in range(batch_size):
            for c in range(channels):
                for h in range(d_out.shape[2]):
                    for w in range(d_out.shape[3]):
                        h_start = h * self.stride
                        h_end = h_start + p_size
                        w_start = w * self.stride
                        w_end = w_start + p_size
                        
                        patch = self.X[b, c, h_start:h_end, w_start:w_end]
                        mask = (patch == np.max(patch))
                        dX[b, c, h_start:h_end, w_start:w_end] += mask * d_out[b, c, h, w]
                        
        return dX


# ==========================================
# 4. FLATTEN LAYER
# ==========================================
class Flatten:
    def forward(self, X):
        self.original_shape = X.shape
        batch_size = X.shape[0]
        return X.reshape(batch_size, -1)

    def backward(self, d_out):
        return d_out.reshape(self.original_shape)


# ==========================================
# 5. FULLY CONNECTED LAYER WITH SIGMOID
# ==========================================
class FullyConnectedWithSigmoid:
    def __init__(self, input_dim, output_dim):
        np.random.seed(42)
        self.weights = np.random.randn(input_dim, output_dim) * 0.1
        self.biases = np.zeros((1, output_dim), dtype=float)
        self.X = None
        self.A2 = None  # Stores A2 = sigmoid(Z2) for backprop derivative
        self.dweights = None
        self.dbiases = None

    def sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))

    def forward(self, X):
        self.X = X
        Z2 = np.dot(X, self.weights) + self.biases
        self.A2 = self.sigmoid(Z2)
        return self.A2

    def backward(self, d_out, learning_rate=0.01):
        dZ2 = d_out * self.A2 * (1 - self.A2)
        self.dweights = np.dot(self.X.T, dZ2)
        self.dbiases = np.sum(dZ2, axis=0, keepdims=True)
        dX = np.dot(dZ2, self.weights.T)

        self.weights -= learning_rate * self.dweights
        self.biases -= learning_rate * self.dbiases

        return dX
# ==========================================
# 6. MAIN EXECUTION & PRINTING SCRIPT
# ==========================================
if __name__ == "__main__":
    sample_input = np.array([[
        [
            [0,   0,   0,   0,   0],
            [0,   0,   0,   0,   0],
            [255, 255, 255, 255, 255],
            [255, 255, 255, 255, 255],
            [255, 255, 255, 255, 255]
        ]
    ]], dtype=float)

    custom_weight = np.array([[
        [[-1,  2, -1],
         [ 0,  0,  0],
         [ 1,  2,  1]]
    ]], dtype=float)
    
    custom_bias = np.array([[0]], dtype=float)

    conv = Conv2D(in_channels=1, out_channels=1, kernel_size=3, custom_weights=custom_weight, custom_bias=custom_bias)
    relu = ReLU()
    maxpool = MaxPool2D(pool_size=2, stride=1)
    flatten = Flatten()
    fc = FullyConnectedWithSigmoid(input_dim=4, output_dim=2)

    print("==================================================")
    print("            FORWARD PASS EXECUTION                ")
    print("==================================================")
    
    out_conv = conv.forward(sample_input)
    print(f"1. Convolution Output (3x3):\n{out_conv[0, 0]}\n")

    out_relu = relu.forward(out_conv)
    print(f"2. ReLU Output (3x3):\n{out_relu[0, 0]}\n")

    out_pool = maxpool.forward(out_relu)
    print(f"3. MaxPool Output (2x2):\n{out_pool[0, 0]}\n")

    out_flat = flatten.forward(out_pool)
    print(f"4. Flattened Output (1x4 vector):\n{out_flat}\n")

    out_fc = fc.forward(out_flat)
    print(f"5. Fully Connected Sigmoid Output (1x2 vector):\n{out_fc}\n")

    d_output_fc = np.array([[1.0, -1.0]])

    print("==================================================")
    print("            BACKWARD PASS EXECUTION               ")
    print("==================================================")

    d_fc = fc.backward(d_output_fc, learning_rate=0.001)
    print(f"1. Gradient after FC Layer Backward (shape {d_fc.shape}):\n{d_fc}\n")

    d_flat = flatten.backward(d_fc)
    print(f"2. Gradient after Flatten Layer Backward shape: {d_flat.shape}\n")

    d_relu = maxpool.backward(d_flat)
    print(f"3. Gradient after MaxPool Layer Backward (3x3):\n{d_relu[0, 0]}\n")

    d_conv = relu.backward(d_relu)
    print(f"4. Gradient after ReLU Layer Backward (3x3):\n{d_conv[0, 0]}\n")

    d_input = conv.backward(d_conv, learning_rate=0.001)
    print(f"5. Final Gradient back to Input Image (5x5):\n{d_input[0, 0]}\n")

    print("==================================================")
    print("         UPDATED WEIGHTS & BIASES (SIGMOID)       ")
    print("==================================================")
    
    # 1. Conv Layer Parameters (W1 & b1)
    print(f"1. Updated Conv Weights (W1_new):\n{conv.weights[0, 0]}\n")
    print(f"   Updated Conv Bias (b1_new):\n{conv.biases}\n")

    # 2. Fully Connected Layer Parameters (W2 & b2)
    print(f"2. Updated FC Weights (W2_new):\n{fc.weights}\n")
    print(f"   Updated FC Biases (b2_new):\n{fc.biases}")

            FORWARD PASS EXECUTION                
1. Convolution Output (3x3):
[[1020. 1020. 1020.]
 [1020. 1020. 1020.]
 [1020. 1020. 1020.]]

2. ReLU Output (3x3):
[[1020. 1020. 1020.]
 [1020. 1020. 1020.]
 [1020. 1020. 1020.]]

3. MaxPool Output (2x2):
[[1020. 1020.]
 [1020. 1020.]]

4. Flattened Output (1x4 vector):
[[1020. 1020. 1020. 1020.]]

5. Fully Connected Sigmoid Output (1x2 vector):
[[1. 1.]]

            BACKWARD PASS EXECUTION               
1. Gradient after FC Layer Backward (shape (1, 4)):
[[0. 0. 0. 0.]]

2. Gradient after Flatten Layer Backward shape: (1, 1, 2, 2)

3. Gradient after MaxPool Layer Backward (3x3):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

4. Gradient after ReLU Layer Backward (3x3):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

5. Final Gradient back to Input Image (5x5):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

         UPDATED WEIGHTS & BIASES (SIGMOID)       
1. Updated Conv Weights (W1_new):
[[-1.  2. -1.]
 